Interview Questions

Cricket Que

In [0]:
%python


 players_data = [
    ("Sachin-IND", 18694, "93/49"),
    ("Ricky-AUS", 11274, "66/31"),
    ("Lara-WI", 10222, "45/21"),
    ("Rahul-IND", 10355, "95/11"),
    ("Jonty-SA", 7051, "43/5"),
    ("Hayden-AUS", 8722, "67/19")
]

players_cols = ["player", "runs", "50s/100s"]

players_df = spark.createDataFrame(players_data, players_cols)

players_df.display()

In [0]:
%python
countries_data = [
    ("IND", "India"),
    ("AUS", "Australia"),
    ("WI", "WestIndies"),
    ("SA", "SouthAfrica")
]

countries_cols = ["SRT", "country"]

countries_df = spark.createDataFrame(countries_data, countries_cols)

countries_df.display()

In [0]:
%python
from pyspark.sql.functions import split

players_dfnew = players_df.withColumn('playername', split(players_df['player'], '-').getItem(0)) \
                           .withColumn('SRT', split(players_df['player'], '-').getItem(1)) \
                             .withColumn('50s', split(players_df['50s/100s'], '/').getItem(0).cast("int")) \
                               .withColumn('100s', split(players_df['50s/100s'], '/').getItem(1).cast("int")) \
                                 .select('playername', 'SRT', 'runs', '50s', '100s')
players_dfnew.display()

In [0]:
%python
from pyspark.sql.functions import col, cast

from pyspark.sql.functions import col, cast
players_dfnew = players_dfnew.withColumn(
    "sum",
    col("50s") + col("100s")
).filter(col("sum") > 90).select("playername", "runs", "SRT", "sum")
players_dfnew.display()

In [0]:
%python
players_dffinal = players_dfnew.join(countries_df, players_dfnew.SRT == countries_df.SRT, 'inner').select("playername", "runs", "country", "sum")
players_dffinal.display()

In [0]:
%sql
CREATE TABLE dim_customerr (
    customer_id INT,
    name STRING,
    city STRING,
    effective_date DATE,
    end_date DATE,
    is_current STRING
);

In [0]:
CREATE TABLE staging_customer (
  customer_id INT,
  name STRING,
  city STRING
);

In [0]:
INSERT INTO dim_customerr VALUES
(1, 'Amit', 'Mumbai', '2023-01-01', '9999-12-31', 'Y'),
(2, 'Rahul', 'Pune', '2023-01-01', '9999-12-31', 'Y')



In [0]:
INSERT INTO staging_customer VALUES
(1, 'Amit', 'Delhi'),      -- changed
(2, 'Rahul', 'Pune'),      -- same
(3, 'Neha', 'Bangalore');  -- new

Step 1 -> Expire Old Records (Update)

In [0]:
MERGE INTO dim_customerr tgt
USING staging_customer src
ON tgt.customer_id = src.customer_id
AND tgt.is_current = 'Y'

WHEN MATCHED AND (
  tgt.name <> src.name OR
  tgt.city <> src.city
)
THEN UPDATE SET
   tgt.end_date = CURRENT_DATE(),
   tgt.is_current = 'N';